# Cascade verifier training (item 1b)

Trains the ResNet18 crop classifier that filters YOLO26n's low-confidence candidates — see `docs/decision_log.md`, 2026-09-13 "Cascade (item 1b) design decisions".

Runs on Colab GPU. All logic lives in `src/`; this notebook only orchestrates (`CLAUDE.md` Section 9).

**Input:** `crops_dataset.zip` — 4808 crops (64x64) mined from **train+val only** by `scripts/09_build_crop_dataset.py`. The test split was never touched. Upload this zip to your Drive next to the videos before running.

**Output:** the trained checkpoint goes to Drive (too large for git, `*.pt` is gitignored); the training history + run manifest get pushed to GitHub.

In [ ]:
!git clone https://github.com/Kametor/object-detection-drone.git
%cd object-detection-drone
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust if you put the zip somewhere else on Drive.
CROPS_ZIP = '/content/drive/MyDrive/object-detection/crops_dataset.zip'

!mkdir -p data
!unzip -q -o "{CROPS_ZIP}" -d data/
!echo "train/person:     $(ls data/crops/train/person | wc -l)"
!echo "train/not_person: $(ls data/crops/train/not_person | wc -l)"
!echo "val/person:       $(ls data/crops/val/person | wc -l)"
!echo "val/not_person:   $(ls data/crops/val/not_person | wc -l)"

In [ ]:
import yaml, torch
print("CUDA available:", torch.cuda.is_available())

# Swap this to try a different verifier config (e.g. configs/26_train_verifier_efficientnet_b0.yaml)
config_path = "configs/24_train_verifier_small_stem.yaml"
config = yaml.safe_load(open(config_path))
config["device"] = "cuda" if torch.cuda.is_available() else "cpu"
# Checkpoint goes to Drive so it survives the runtime being recycled.
config["checkpoint_path"] = "/content/drive/MyDrive/object-detection/resnet18_verifier_small_stem.pt"
yaml.safe_dump(config, open(config_path, "w"), sort_keys=False)
print(open(config_path).read())


In [ ]:
!python scripts/10_train_verifier.py --config {config_path}


In [ ]:
import getpass

gh_token = getpass.getpass("GitHub personal access token: ")

!git config user.email "you@example.com"
!git config user.name "Colab"
# Only history + manifest: the checkpoint lives on Drive (*.pt is gitignored).
!git add results/cascade_verifier/training_history.json results/manifests
!git commit -m "Add cascade verifier training run (Colab)"
!git pull origin main --no-edit --no-rebase
!git push https://{gh_token}@github.com/Kametor/object-detection-drone.git main